In [4]:
# Vancouver 311 Service Request - Data Cleaning
# Author: Naini Singh
# Date: 22 June 2026
# Purpose: Filter and prepare 311 data for spatial hotspot analysis in ArcGIS Pro
# Data: City of Vancouver Open Data Portal - https://opendata.vancouver.ca/explore/dataset/3-1-1-service-requests/export/?disjunctive.service_request_type&disjunctive.status&disjunctive.channel&disjunctive.local_area&disjunctive.department&disjunctive.closure_reason&location=10,49.20636,-123.20012&basemap=jawg.streets 

import pandas as pd
import os

print("Libraries loaded successfully")

Libraries loaded successfully


In [5]:
# Loading Data
file_path = r"C:\Users\shaha\Desktop\Projects\GIS\Project - Vancouver-3-1-1 Hotspot Analysis\Data\3-1-1-service-requests.csv"

print("Loading dataset:")

df = pd.read_csv(
    file_path,
    sep=';',
    encoding='utf-8-sig',
    low_memory=False
)

print(f"Dataset loaded successfully")
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nColumn names:")
for col in df.columns:
    print(f"  - {col}")

Loading dataset:
Dataset loaded successfully
Total rows: 1,048,575
Total columns: 13

Column names:
  - Department
  - Service request type
  - Status
  - Closure reason
  - Service request open timestamp
  - Service request close date
  - Last modified timestamp
  - Address
  - Local area
  - Channel
  - Latitude
  - Longitude
  - geom,


In [6]:
# Inspecting Data
print("FIRST 3 ROWS:")
print(df.head(3).to_string())

print("\nDATA TYPES:")
print(df.dtypes)

print("\nMISSING VALUES PER COLUMN:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df)

FIRST 3 ROWS:
                  Department          Service request type Status                  Closure reason Service request open timestamp Service request close date    Last modified timestamp              Address    Local area Channel  Latitude   Longitude                        geom,
0  ENG - Sanitation Services  Missed Green Bin Pickup Case  Close                Service provided      2022-09-28T20:57:54-07:00                 2022-09-30  2022-09-30T05:46:47-07:00                  NaN  South Cambie     WEB       NaN         NaN                            ,
1         311 Contact Centre   City Services Feedback Case  Close  Reviewed and no action planned      2022-09-28T21:01:10-07:00                 2023-04-28  2023-04-28T13:08:07-07:00  1800 SPYGLASS PLACE      Fairview   Phone  49.27044 -123.114881  49.27044035727,-123.1148813
2  ENG - Sanitation Services  Missed Green Bin Pickup Case  Close                Service provided      2022-09-28T21:02:38-07:00                 2022-09-30

In [11]:
#FILTER TO INFRASTRUCTURE DEPARTMENTS 
infrastructure_depts = [
    'ENG - Streets Operations',
    'ENG - Waterworks Operations',
    'ENG - Traffic and Electrical Operations and Design',
    'PR - Urban Forestry'
]

df_infra = df[df['Department'].isin(infrastructure_depts)].copy()
print(f"After infrastructure filter: {len(df_infra):,} rows")

#EXTRACT YEAR DIRECTLY FROM STRING
# Timestamp format is: 2023-04-15T... so first 4 characters = year
df_infra['Year'] = df_infra['Service request open timestamp'].str[:4]
df_infra['Month'] = df_infra['Service request open timestamp'].str[5:7]

print(f"\nYear distribution (all years):")
print(df_infra['Year'].value_counts().sort_index())

#FILTER TO 2023-2025
df_infra = df_infra[df_infra['Year'].isin(['2023', '2024', '2025'])].copy()
print(f"\nAfter 2023-2025 filter: {len(df_infra):,} rows")

print(f"\nRows per department:")
print(df_infra['Department'].value_counts())

After infrastructure filter: 155,800 rows

Year distribution (all years):
Year
2022    36794
2023    35435
2024    36426
2025    33676
2026    13469
Name: count, dtype: int64

After 2023-2025 filter: 105,537 rows

Rows per department:
Department
ENG - Streets Operations                              41822
PR - Urban Forestry                                   31387
ENG - Traffic and Electrical Operations and Design    20340
ENG - Waterworks Operations                           11988
Name: count, dtype: int64


In [12]:
#SPLIT 1: RECORDS WITH COORDINATES 
# For kernel density point analysis in ArcGIS Pro
df_with_coords = df_infra[
    df_infra['Latitude'].notna() & 
    df_infra['Longitude'].notna()
].copy()

print(f"Records WITH coordinates: {len(df_with_coords):,}")

#SPLIT 2: RECORDS BY NEIGHBOURHOOD 
# For choropleth map (includes records without coords)
df_by_neighbourhood = df_infra[
    df_infra['Local area'].notna() & 
    (df_infra['Local area'] != '')
].copy()

#Aggregate by neighbourhood and department
neighbourhood_summary = df_by_neighbourhood.groupby(
    ['Local area', 'Department', 'Year']
).size().reset_index(name='Request_Count')

print(f"Records WITH local area: {len(df_by_neighbourhood):,}")
print(f"\nNeighbourhood summary rows: {len(neighbourhood_summary):,}")
print(f"\nTop 10 neighbourhoods by total requests:")
top_areas = df_by_neighbourhood['Local area'].value_counts().head(10)
print(top_areas)

Records WITH coordinates: 90,877
Records WITH local area: 103,025

Neighbourhood summary rows: 264

Top 10 neighbourhoods by total requests:
Local area
Kensington-Cedar Cottage    8568
Downtown                    8441
Kitsilano                   7300
Renfrew-Collingwood         7142
Hastings-Sunrise            6294
Grandview-Woodland          5547
Riley Park                  5508
Mount Pleasant              5318
Sunset                      5154
Fairview                    4601
Name: count, dtype: int64


In [13]:
import os

#SET OUTPUT PATH
output_path = r"C:\Users\shaha\Desktop\Projects\GIS\Project - Vancouver-3-1-1 Hotspot Analysis\Data"

#EXPORT 1: POINTS FILE FOR ARCGIS PRO
coords_file = os.path.join(output_path, "vancouver_311_infrastructure_points.csv")

df_with_coords[[
    'Department',
    'Service request type', 
    'Status',
    'Local area',
    'Year',
    'Month',
    'Latitude',
    'Longitude'
]].to_csv(coords_file, index=False)

print(f"Points file saved: {len(df_with_coords):,} rows")
print(f"Location: {coords_file}")

#EXPORT 2: NEIGHBOURHOOD SUMMARY FOR CHOROPLETH
neighbourhood_file = os.path.join(output_path, "vancouver_311_neighbourhood_summary.csv")

neighbourhood_summary.to_csv(neighbourhood_file, index=False)

print(f"\nNeighbourhood summary saved: {len(neighbourhood_summary):,} rows")
print(f"Location: {neighbourhood_file}")

print("\n--- DATA QUALITY SUMMARY ---")
print(f"Original dataset:              1,048,575 rows")
print(f"After infrastructure filter:     155,800 rows")
print(f"After 2023-2025 filter:          105,537 rows")
print(f"Points with coordinates:          90,877 rows")
print(f"Records with neighbourhood:      103,025 rows")
print(f"\nReady for ArcGIS Pro import.")

Points file saved: 90,877 rows
Location: C:\Users\shaha\Desktop\Projects\GIS\Project - Vancouver-3-1-1 Hotspot Analysis\Data\vancouver_311_infrastructure_points.csv

Neighbourhood summary saved: 264 rows
Location: C:\Users\shaha\Desktop\Projects\GIS\Project - Vancouver-3-1-1 Hotspot Analysis\Data\vancouver_311_neighbourhood_summary.csv

--- DATA QUALITY SUMMARY ---
Original dataset:              1,048,575 rows
After infrastructure filter:     155,800 rows
After 2023-2025 filter:          105,537 rows
Points with coordinates:          90,877 rows
Records with neighbourhood:      103,025 rows

Ready for ArcGIS Pro import.


In [14]:
# DATA QUALITY & METHODOLOGY NOTES


print("""
VANCOUVER 311 INFRASTRUCTURE HOTSPOT ANALYSIS
Data Cleaning Summary
Author: Naini Singh
Date: June 2026


SOURCE DATA

Dataset:    City of Vancouver 3-1-1 Service Requests
Source:     opendata.vancouver.ca
Format:     CSV (semicolon delimited)
Rows:       1,048,575
Columns:    13
Period:     2022-2026

FILTERING DECISIONS

1. DEPARTMENT FILTER
   Retained 4 infrastructure departments only:
   - ENG - Streets Operations
   - ENG - Waterworks Operations  
   - ENG - Traffic and Electrical Operations and Design
   - PR - Urban Forestry
   Rationale: Focus on physical infrastructure maintenance
   relevant to municipal operations and asset management.

2. DATE FILTER
   Retained 2023-2025 only (3 complete years)
   Rationale: Excludes partial years 2022 and 2026
   for consistent temporal comparison.

DATA QUALITY FINDINGS
456
Missing coordinates: 66.3% of original dataset
Missing local area:  21.7% of original dataset
Approach: Dataset split into two outputs:
   - Points file (with coords) for KDE analysis
   - Neighbourhood summary for choropleth mapping
   This dual approach maximises usable data while
   maintaining spatial accuracy.

OUTPUT FILES
------------
1. vancouver_311_infrastructure_points.csv
   Rows: 90,877 | Use: Kernel Density in ArcGIS Pro

2. vancouver_311_neighbourhood_summary.csv  
   Rows: 264    | Use: Choropleth map + Power BI

KEY FINDING (PRELIMINARY)
--------------------------
Top infrastructure demand neighbourhoods (2023-2025):
1. Kensington-Cedar Cottage: 8,568 requests
2. Downtown:                 8,441 requests
3. Kitsilano:                7,300 requests
4. Renfrew-Collingwood:      7,142 requests
5. Hastings-Sunrise:         6,294 requests
""")


VANCOUVER 311 INFRASTRUCTURE HOTSPOT ANALYSIS
Data Cleaning Summary
Author: Naini Singh
Date: June 2026


SOURCE DATA

Dataset:    City of Vancouver 3-1-1 Service Requests
Source:     opendata.vancouver.ca
Format:     CSV (semicolon delimited)
Rows:       1,048,575
Columns:    13
Period:     2022-2026

FILTERING DECISIONS

1. DEPARTMENT FILTER
   Retained 4 infrastructure departments only:
   - ENG - Streets Operations
   - ENG - Waterworks Operations  
   - ENG - Traffic and Electrical Operations and Design
   - PR - Urban Forestry
   Rationale: Focus on physical infrastructure maintenance
   relevant to municipal operations and asset management.

2. DATE FILTER
   Retained 2023-2025 only (3 complete years)
   Rationale: Excludes partial years 2022 and 2026
   for consistent temporal comparison.

DATA QUALITY FINDINGS
456
Missing coordinates: 66.3% of original dataset
Missing local area:  21.7% of original dataset
Approach: Dataset split into two outputs:
   - Points file (with coord

In [15]:
#SAVE SAMPLE FOR GITHUB (5000 rows) 
sample_file = r"C:\Users\shaha\Desktop\Projects\GIS\Project - Vancouver-3-1-1 Hotspot Analysis\Data\vancouver_311_points_SAMPLE.csv"

df_with_coords.sample(5000, random_state=42).to_csv(sample_file, index=False)

print("Sample file saved: 5,000 rows")
print("Full dataset (90,877 rows) remains on local machine for ArcGIS Pro")

Sample file saved: 5,000 rows
Full dataset (90,877 rows) remains on local machine for ArcGIS Pro
